In [ ]:
# les notebooks vivent dans notebooks/ : on se replace à la racine PythonIPM
# pour que les chemins relatifs (EHCVM/, sorties/) et les imports du pipeline marchent
import os, sys
from pathlib import Path

RACINE = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(RACINE)
sys.path.insert(0, str(RACINE / "pipeline"))


In [4]:
"""Étape 01 du pipeline IPM — matrice situationnelle X (EHCVM 2021).

Première étape de la méthode Alkire-Foster (chapitre 4 du guide ODD) : X contient la
*situation* de chaque ménage dans chaque indicateur, sans aucun seuil de privation.
Les étapes suivantes (seuils z, matrice de privation g0, pondérations, H/A/M0) feront
l'objet des fichiers 02_, 03_ et 04_.

Une ligne de X = un ménage (12 965), conformément au document méthodologique de l'IPM-CI :
le score de privation se calcule au niveau du ménage, chaque ménage comptant ensuite pour
sa taille dans les indices.

Chaque indicateur suit SOIT la proposition nationale, SOIT l'application du PNUD — le choix
est porté par la constante INDICATEURS ci-dessous et détaillé dans METHODOLOGIE.md :

    proposition nationale : fréquentation scolaire, année de scolarité, alphabétisation,
                            état civil, assurance maladie, électricité, énergie de cuisson,
                            emploi
    PNUD                  : logement, eau potable, toilettes, biens d'équipement

Usage :
    python 01_matrice_situationnelle.py            # construit X et écrit le .dta + le log
    python 01_matrice_situationnelle.py --check    # auto-contrôle sur un mini-jeu de données
"""
import logging
import sys
import time
from collections import namedtuple
from pathlib import Path

import numpy as np
import pandas as pd

import dictionnaire_ehcvm as dico

DATA = Path("EHCVM")
SORTIE = Path("matrice_situationnelle_ehcvm2021.dta")
JOURNAL = Path("01_matrice_situationnelle.log")

CLE = ["grappe", "menage", "vague"]

NATIONALE = "proposition nationale"
PNUD = "application PNUD"
ABSENT = "sans source dans l'EHCVM"

Indicateur = namedtuple("Indicateur", "dimension libelle source definition situation eligibilite")

# Carte officielle des indicateurs : une entrée par ligne du tableau de référence.
# `situation`  = colonnes de X qui portent la situation du ménage
# `eligibilite` = colonne de X qui dit combien de membres sont concernés (None si tous le sont)
INDICATEURS = [
    Indicateur("Education", "Fréquentation scolaire", NATIONALE,
               "Le ménage a un enfant de 6-16 ans qui ne fréquente actuellement pas",
               ["enfants_6_16_non_scolarises"], "enfants_6_16"),
    Indicateur("Education", "Année de scolarité", NATIONALE,
               "Aucun membre du ménage âgé de 17-95 ans n'a complété 10 années d'études",
               ["annees_etudes_max"], "membres_17_95"),
    Indicateur("Education", "Alphabétisation", NATIONALE,
               "Un membre du ménage de 17-49 ans ne sait pas lire ou écrire (français)",
               ["membres_17_49_alphabetises"], "membres_17_49"),
    Indicateur("Education", "Déclaration d'état civil", NATIONALE,
               "Un membre de 5-15 ans n'a pas d'acte de naissance ou n'est pas déclaré",
               ["enfants_5_15_sans_acte"], "enfants_5_15"),
    
    Indicateur("Sante", "Assurance maladie", NATIONALE,
               "Aucun membre du ménage n'est couvert par une assurance maladie",
               ["membres_assures"], "taille_menage"),
    
    Indicateur("Emploi", "Chômage", NATIONALE,
               "Un membre du ménage âgé de 17-40 ans est au chômage",
               ["chomeurs_17_40"], "membres_17_40"),

    Indicateur("Conditions de vie", "Electricité", NATIONALE,
               "La source d'éclairage n'est pas : électricité, groupe électrogène ou solaire",
               ["source_eclairage"], None),
    Indicateur("Conditions de vie", "Logement", PNUD,
               "Sol en matériaux naturels et/ou toit et/ou murs en matériaux naturels "
               "ou rudimentaires",
               ["materiau_toit", "materiau_mur", "materiau_sol"], None),
    Indicateur("Conditions de vie", "Eau potable", PNUD,
               "Pas d'eau améliorée (ODD), ou eau à 30 minutes ou plus à pied aller-retour",
               ["source_eau_boisson_seche", "temps_aller_source_seche"], None),
    Indicateur("Conditions de vie", "Energie de cuisson", NATIONALE,
               "Le ménage n'utilise pas d'énergie propre pour la cuisson (électricité et gaz)",
               ["combustible_principal"], None),
    Indicateur("Conditions de vie", "Toilette", PNUD,
               "Installations sanitaires non améliorées (ODD), ou améliorées mais partagées",
               ["type_sanitaire", "sanitaire_partage"], None),
    Indicateur("Conditions de vie", "Biens d'équipement", PNUD,
               "Le ménage ne possède qu'un seul bien parmi radio, télévision, téléphone, "
               "ordinateur, charrette, vélo, moto, réfrigérateur, et pas de voiture",
               ["nb_equipements", "possede_voiture"], None),
]

# tranches d'âge, telles que définies dans le tableau de référence
TRANCHES = {
    "scolarisation": (6, 16),     # nationale
    "annees_etudes": (17, 95),    # nationale
    "alphabetisation": (17, 49),  # nationale
    "chomage": (17, 40),          # nationale
    "acte_naissance": (5, 15),    # nationale
}

# années d'études accomplies AVANT d'entrer dans le niveau (questions 2.29 et 2.14)
ANNEES_AVANT_NIVEAU = {
    1: 0,   # maternelle
    2: 0,   # primaire                 (1re a 6e annee)
    3: 6,   # secondaire 1 general     (6e a 3e -> 7e a 10e annee)
    4: 6,   # secondaire 1 technique
    5: 10,  # secondaire 2 general     (2nde a Tle)
    6: 10,  # secondaire 2 technique
    7: 13,  # post-secondaire
    8: 13,  # superieur
}

# question 11.52 : deux principaux combustibles, ordonnés (rang 1 = principal)
COMBUSTIBLES = ["combustible_bois_ramasse", "combustible_bois_achete", "combustible_charbon",
                "combustible_gaz", "combustible_electricite", "combustible_petrole",
                "combustible_dechets_animaux", "combustible_autre"]

# Biens d'équipement, définition PNUD : « radio, télévision, téléphone, ordinateur,
# charrette, vélo, moto ou réfrigérateur ». Un bien = une ligne, même s'il correspond à
# plusieurs codes de la section 12 (le téléphone peut être fixe ou portable).
# Écart assumé : la CHARRETTE ne figure pas dans les 45 biens de la section 12 de l'EHCVM,
# le décompte porte donc sur 7 biens et non 8 (voir METHODOLOGIE.md).
EQUIPEMENTS_PNUD = {
    "radio": [19],
    "television": [20],
    "telephone": [34, 35],       # fixe ou portable
    "ordinateur": [37],
    "bicyclette": [30],
    "moto": [29],
    "refrigerateur": [16],
}
VOITURE = 28                     # le camion n'existe pas non plus dans la section 12

ITEMS_FIES = ["fies_inquietude", "fies_pas_sain", "fies_peu_varie", "fies_saute_repas",
              "fies_mange_moins", "fies_plus_de_nourriture", "fies_faim",
              "fies_journee_sans_manger"]

COLONNES_MENAGE = [
    # conditions de vie
    "source_eclairage", "materiau_toit", "materiau_mur", "materiau_sol",
    "source_eau_boisson_seche", "temps_aller_source_seche",
    "combustible_principal", "type_sanitaire", "sanitaire_partage",
    # identification, pondération et désagrégation
    "id_menage", "ponderation_menage", "taille_menage", "region", "milieu",
]

logger = logging.getLogger("ipm.matrice_situationnelle")


def configurer_logs(fichier=JOURNAL, niveau=logging.INFO):
    """Console (niveau demandé) + fichier de log détaillé (DEBUG)."""
    logger.setLevel(logging.DEBUG)
    logger.handlers.clear()

    console = logging.StreamHandler(sys.stdout)
    console.setLevel(niveau)
    console.setFormatter(logging.Formatter("%(message)s"))
    logger.addHandler(console)

    if fichier:
        journal = logging.FileHandler(fichier, mode="w", encoding="utf8")
        journal.setLevel(logging.DEBUG)
        journal.setFormatter(logging.Formatter("%(asctime)s  %(levelname)-7s  %(message)s"))
        logger.addHandler(journal)
        logger.debug("journal ouvert : %s", fichier)


def part(effectif, total):
    """« 1 204 (2,6 %) » — un effectif ne se lit jamais sans son dénominateur."""
    if not total:
        return f"{effectif:,}".replace(",", " ")
    return f"{effectif:,}".replace(",", " ") + f" ({effectif / total:.1%})"

In [6]:
# --------------------------------------------------------------------------- #
# 1. chargement
# --------------------------------------------------------------------------- #
def charger_base(fichier, colonnes=None):
    """Lit un .dta en codes bruts (pas les libellés) et applique les noms explicites."""
    noms = dico.BASES[fichier]
    codes = {v: k for k, v in noms.items()}
    a_lire = [codes[c] for c in colonnes] if colonnes else list(noms)

    debut = time.perf_counter()
    d = pd.read_stata(DATA / fichier, columns=a_lire, convert_categoricals=False)
    d = d.rename(columns=noms)

    logger.info("%-32s %7d lignes x %3d colonnes  (%.1f s)",
                fichier, len(d), d.shape[1], time.perf_counter() - debut)
    logger.debug("%s : colonnes lues = %s", fichier, ", ".join(d.columns))
    return d

In [8]:
def charger_bases():
    """Les 4 bases utiles, avec nettoyage des lignes vides de la base FIES."""
    logger.info("--- 1. chargement des bases (codes bruts, colonnes du dictionnaire) ---")
    individus = charger_base("Base_Individus.dta")
    menages = charger_base("Base_Menage.dta")
    avoirs = charger_base("Base_avoirs_du_menage.dta")
    fies = charger_base("Base_securite_alimentaire.dta")

    vides = fies[CLE].isna().all(axis=1).sum()
    fies = fies.dropna(subset=CLE)
    logger.info("base FIES : %d lignes entièrement vides supprimées -> %d lignes", vides, len(fies))

    for nom, base in [("individus", individus), ("ménages", menages),
                      ("avoirs", avoirs), ("FIES", fies)]:
        logger.info("  %-10s : %6d ménages distincts", nom, len(base[CLE].drop_duplicates()))
    return individus, menages, avoirs, fies

In [9]:
# --------------------------------------------------------------------------- #
# 2. situations individuelles
# --------------------------------------------------------------------------- #
def calculer_age(ind):
    """Âge = année d'enquête (vague 1 = 2021, vague 2 = 2022) - année de naissance."""
    logger.info("--- 2.1 âge ---")
    annee_enquete = np.where(ind.vague == 1, 2021, 2022)
    age = pd.Series(annee_enquete - ind.annee_naissance, index=ind.index)

    logger.info("année de naissance renseignée : %s",
                part(ind.annee_naissance.notna().sum(), len(ind)))
    logger.info("âge déclaré (1.04a) renseigné : %s -> utilisé en priorité",
                part(ind.age_declare.notna().sum(), len(ind)))

    ind["age"] = ind.age_declare.fillna(age)
    aberrants = ((ind.age < 0) | (ind.age > 110)).sum()
    logger.info("âge calculé pour %s ; âges hors bornes (0-110 ans) : %d",
                part(ind.age.notna().sum(), len(ind)), aberrants)
    logger.debug("distribution de l'âge :\n%s", ind.age.describe().round(1).to_string())
    return ind